# Pose predictors

## File Handling
To run predictions a `RobotEnvironment` object and a `HeadsetData` object is needed, those can be loaded from folders or created.

### Creation of RobotEnvironment and HeadsetData
Those 2 datatypes can be created from an GatheredRobotData object and a .vrs file respectively.

In [ ]:
%load_ext autoreload
%autoreload 2
print(__debug__)

import logging
logging.basicConfig(level=logging.INFO)

import seaborn as sns
sns.set_theme(style="whitegrid", context="paper", font_scale=1)


import numpy as np
import random, torch, os, cv2

seed = 1

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
cv2.setRNGSeed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(seed)

import matplotlib.pyplot as plt

from pose_estimation import *

In [ ]:
from shared.complete_robot_scan import CompleteRobotScan
robot_data_folder_location = "../example_datasets/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1_sitting_20fps.vrs"

#robot_data_folder_location = "../example_datasets/small_aruco_2"
#vrs_file_location = "../example_datasets/small_aruco_2_2.vrs"


robot_data = CompleteRobotScan.from_folder(robot_data_folder_location)
robot_env = Scanned3dEnvironment.from_gathered_robot_data(
        robot_data = robot_data,
        number_of_sampled_datapoints=10,
        sample_datapoints_based_on_aruco_corectness = False,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=False)
)

labeled_headset_data = create_robot_bound_headset_data(
        headset_data = HeadsetData.from_vrs_file(vrs_file_location),
        robot_data = robot_data
)

visualize_loaded_data = False

if visualize_loaded_data:
    visualize_robot_camera_environment_combo(robot_env=robot_env, headset_data=labeled_headset_data)


## Testing Predictors

### Creating Predictors
Now an `PosePredictor` can be created. An `PosePredictor` instance is build upon an `RobotEnvironment` instance and can predict positions from headset-images.

In [ ]:
#pne_optimizer = PyposePNEOptimizer(PyposePnEOptimizerConfig())
#pne_optimizer = PnEDeltaPoseLBFGSOptimizer(time_tracker=tt_pne)

predictor = EllipsoidPredictor(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Rotate180Deg],
            extract_and_match=ExtractAndLightGlue(),
            ransac_config=pose_estimation_ransaac_config_less_precise,
        ),
        pne_optimizer=PnEDeltaPoseAdamOptimizer(),
        ellipsoid_refinement_at_res=(1400, 1400),
        cam1_segmenter=SAM3Segmenter(Sam3Prompt()),
        cam2_segmenter=YOLOv26Segmenter("yoloe-26l-seg.pt"),
        matching_config=GaussianMatchingConfig(dummy_value=0.01),
        visualize_pne_optimisation=False,
        visualize_environment_generation=True
)

In [ ]:
ge = FastGrippingError(points=robot_env.robot_xyz_images.reshape(-1,3), intrinsics=labeled_headset_data.intrinsic_cam_mtx, visualize=True)

grade = PredictionOnDataset(
    predictor=predictor,
    headset_data=labeled_headset_data
)

i, pred, label = grade.comparable_poses[0]

pixels = sample_pixel_neighborhood(center=(550,700))

fig, ax = plt.subplots(1,1)
ax.imshow(labeled_headset_data.bgr_image_s[i])
ax.scatter(pixels[:, 0], pixels[:,1], c='red', marker='o', s=2, label='Type A')

print(ge.calculate_gripping_differences_4_pixels(base_t_cam_s=np.array([pred, label]), pixels_batch=np.array([pixels, pixels])))

### Ellipsoid fitting

In [ ]:
import open3d as o3d

simple_fitter = SimpleEllipsoidFitter(visualize=False)
mvee_fitter = MVEEEllipsoidFitter(visualize=False, contamination=0.0)
ls_fitter = LeastShellDistanceEllipsoidFitter(visualize=False, size_penalty=0.95)

fitters = [simple_fitter, mvee_fitter, ls_fitter]

n_samples = 1000

points_x = np.random.rand(n_samples)-0.5
points_y = np.random.rand(n_samples)-0.5
points_z = -points_x**2 - points_y**2

pc = np.column_stack([points_x, points_y, points_z]) + 0.01 * np.random.rand(n_samples, 3)
print(f"pc: {pc.shape}")

for fitter in fitters:
    base_t_ellipsoid, primal_quadratic = fitter.fit_ellipsoid(points=pc)
    ls = create_ellipsoid_lineset(base_t_ellipsoid, primal_quadratic)

    pcd1 = o3d.geometry.PointCloud()
    pcd1.points = o3d.utility.Vector3dVector(pc)
    pcd1.paint_uniform_color([1,0,0])

    frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.05)
    frame.transform(base_t_ellipsoid)

    to_vis = [ls, pcd1, frame]
    o3d.visualization.draw_geometries(to_vis, f"Ellipsoid fit visualization", 1200, 1200)